In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
Hs_immune_dir="~/chrombpnet/chrombpnet_use_snakefile/Hs_C10white_blood_cells/output/predict/finemo"
TS_immune_dir="~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C24white_blood_cells/output/predict/finemo"
output_dir="~/chrombpnet/Hs_TS_immune_chrombpnet/motif_cooccurrence"

In [ ]:
#树鼩的C24.3之前分析过cooccurrence
immune_dirs = {
    Hs_immune_dir: ["C10.1","C10.2","C10.3","C10.4","C10.5","C10.6","C10.7","C10.8","C10.9","C10.10","C10.11","C10.12","C10.13","C10.14"],
    TS_immune_dir: ["C24.1","C24.2","C24.4","C24.5","C24.6","C24.7","C24.8","C24.9","C24.10","C24.11","C24.12","C24.13","C24.14","C24.15","C24.16","C24.17","C24.18","C24.19","C24.20","C24.21","C24.22","C24.23"]
}

In [ ]:
Hs_immune_dir="~/chrombpnet/chrombpnet_use_snakefile/Hs_C3Erythroid/output/predict/finemo"
output_dir="~/chrombpnet/Hs_TS_immune_chrombpnet/motif_cooccurrence"

In [ ]:
immune_dirs = {
    Hs_immune_dir: ["C3.1","C3.3"]
}

In [ ]:
def calculate_cooccurrence(species, celltype, predict_dir, output_dir):

    hit_file = os.path.join(predict_dir,celltype,"hit_call","hits.tsv")

    print(f"Processing {celltype}")

    if not os.path.exists(hit_file):
        print(f"  hits.tsv not found, skip: {hit_file}")
        return None

    # =========================
    # 1. 读取 Finemo hits
    # =========================
    hits = pd.read_csv(hit_file,sep="\t")

    print(f"  motif hits: {len(hits)}")
    print(f"  peaks with motif hits: {hits['peak_id'].nunique()}")
    print(f"  motifs: {hits['motif_name'].nunique()}")

    # =========================
    # 2. peak × motif count matrix
    # =========================
    motif_matrix = pd.crosstab(hits["peak_id"],hits["motif_name"])

    # =========================
    # 3. 转成 presence / absence
    # =========================
    motif_binary = (motif_matrix > 0).astype(int)

    # =========================
    # 4. motif × motif co-occurrence
    # =========================
    cooccurrence_matrix = (motif_binary.T.dot(motif_binary))

    # =========================
    # 5. 保存结果
    # =========================
    motif_matrix.to_csv(
        os.path.join(
            output_dir,
            f"{species}_{celltype}_Allpeaks_motif_presence_matrix.csv"
        )
    )

    cooccurrence_matrix.to_csv(
        os.path.join(
            output_dir,
            f"{species}_{celltype}_Allpeaks_motif_cooccurrence_matrix.csv"
        )
    )

    return cooccurrence_matrix


In [ ]:
# =========================
# 批量处理
# =========================

all_cooccurrence = {}

for base_dir, celltypes in immune_dirs.items():

    if base_dir == Hs_immune_dir:
        species = "human"
    elif base_dir == TS_immune_dir:
        species = "treeshrew"

    if species not in all_cooccurrence:
        all_cooccurrence[species] = {}

    for celltype in celltypes:

        result = calculate_cooccurrence(
            species=species,
            celltype=celltype,
            predict_dir=base_dir,
            output_dir=os.path.join(output_dir,species)
        )

        if result is not None:
            all_cooccurrence[species][celltype] = result

print("\nFinished.")

In [ ]:
# =========================
# 批量处理
# =========================

all_cooccurrence = {}

for base_dir, celltypes in immune_dirs.items():

    if base_dir == Hs_immune_dir:
        species = "human"
    elif base_dir == TS_immune_dir:
        species = "treeshrew"

    if species not in all_cooccurrence:
        all_cooccurrence[species] = {}

    for celltype in celltypes:

        result = calculate_cooccurrence(
            species=species,
            celltype=celltype,
            predict_dir=base_dir,
            output_dir=os.path.join(output_dir,species)
        )

        if result is not None:
            all_cooccurrence[species][celltype] = result

print("\nFinished.")